# Demo: BERTweet Sentiment Inference
This notebook loads the fine-tuned model from Hugging Face, evaluates test accuracy on the Sentiment140 test split, and predicts sentiment for custom user text.

The dataset split logic matches the Full_Tuning notebook:
- Filter labels to only `0` and `4`
- Relabel to binary: `0 -> 0 (negative)`, `4 -> 1 (positive)`
- Normalize mentions and URLs
- Split with seed `3` using 90/10, then split the 10% equally into validation/test

In [1]:
# Keep Colab's default torch build and install required NLP libs.
%pip install -U transformers datasets evaluate
%pip uninstall -y torchvision torchaudio
print("If this is a fresh runtime, proceed. If not, Runtime > Restart session, then run all cells.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.1

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
import evaluate
import torch
import re

# Choose which seed to load from your model repo.
seed_subfolder = "seed3"  # change to "seed2" or "seed3" if needed
model_name = "BrianLeung/bertweet-sentiment140-full-finetune"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    subfolder=seed_subfolder,
).to(device)

#tokenizer = AutoTokenizer.from_pretrained(
   # model_name,
  #  subfolder=seed_subfolder,
 #)

tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base")

model.eval()

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/936 [00:00<?, ?B/s]

seed3/model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(130, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [3]:
dataset = load_dataset("BrianLeung/sentiment140-csv", encoding="ISO-8859-1")

# The dataset may expose different split names; use train if available.
if "train" in dataset:
    temp = dataset["train"]
else:
    first_split_name = list(dataset.keys())[0]
    temp = dataset[first_split_name]

# Match Full_Tuning preprocessing exactly.
temp = temp.remove_columns(["ID", "Date", "Flag", "User"])
temp = temp.filter(lambda x: x["Label"] in [0, 4])

def relabel(batch):
    labels = batch["Label"]
    new_labels = [0 if l == 0 else 1 if l == 4 else None for l in labels]
    if any(l is None for l in new_labels):
        raise ValueError("Unexpected label in batch")
    return {"Label": new_labels}

def replace(text):
    text = re.sub(r"@\w+", "@USER", text)
    text = re.sub(r"http\S+|www\.\S+", "HTTPURL", text)
    return text

def clean_text(batch):
    return {"Text": [replace(t) for t in batch["Text"]]}

temp = temp.map(relabel, batched=True)
temp = temp.map(clean_text, batched=True)
temp = temp.train_test_split(test_size=0.1, seed=3)
test_valid = temp["test"].train_test_split(test_size=0.5, seed=3)

dataset_dict = {
    "train": temp["train"].shuffle(seed=3).select(range(1)),
    "validation": test_valid["train"].select(range(1)),
    "test": test_valid["test"],
}

for split_name, split_data in dataset_dict.items():
    print(split_name, len(split_data))

Sentiment140.csv:   0%|          | 0.00/239M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1600000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1600000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600000 [00:00<?, ? examples/s]

train 1
validation 1
test 80000


In [4]:
label2id = {"negative": 0, "positive": 1}
id2label = {0: "negative", 1: "positive"}

# Match Full_Tuning truncation behavior.
max_positions = model.config.max_position_embeddings
max_input_len = max_positions - 2
if max_input_len <= 0:
    raise ValueError(f"max_input_len={max_input_len} is invalid")

def predict_batch(texts, max_length=max_input_len):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        logits = model(**encoded).logits
        preds = torch.argmax(logits, dim=-1)
    return preds.cpu().tolist()

test_ds = dataset_dict["test"]
batch_size = 64
all_preds = []

for i in range(0, len(test_ds), batch_size):
    batch_texts = test_ds[i:i + batch_size]["Text"]
    batch_preds = predict_batch(batch_texts)
    all_preds.extend(batch_preds)

accuracy_metric = evaluate.load("accuracy")
test_accuracy = accuracy_metric.compute(
    predictions=all_preds,
    references=test_ds["Label"],
)["accuracy"]

print(f"Test accuracy: {test_accuracy:.4f}")

Test accuracy: 0.8910


In [5]:
def classify_text(text):
    cleaned = replace(text)
    pred_id = predict_batch([cleaned])[0]
    label = id2label[pred_id]
    return {
        "text": text,
        "cleaned_text": cleaned,
        "predicted_label": label,
        "predicted_id": pred_id,
    }

# Edit this input and run the cell to classify your own sentence.
user_text = "I love how fast and reliable this model is!"
result = classify_text(user_text)
print(result)

{'text': 'I love how fast and reliable this model is!', 'cleaned_text': 'I love how fast and reliable this model is!', 'predicted_label': 'positive', 'predicted_id': 1}
